Example to verify that the added openmp will produce the same spectra as the serial calculation. It will also calculate the difference between using one and 4 threads. 

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np

#right now I guess the best way to do this is to either have the serial one locally next to this version, 
#or to use the older version of modbhjet (we can try to morph these serial versions together later - this is just from mine: )

#this is the same method as how I set up kariba for the tutorial: 

# git clone --branch modularUpdate https://github.com/rubyduncan/modBHJet.git ../modBHJet-serial

# python -m pip wheel --no-build-isolation --no-deps \
#     ../modBHJet-serial --wheel-dir ../modBHJet-serial/dist


openmp_root = Path("..").resolve()

#need to then either set this directly with what you have locally or just download another version from github
# serial_root = openmp_root.parent / "modBHJet-serial"


In [ ]:
calculation = """
import json
import time
import numpy as np
from bhjet import BHJet, BLJet

energy_eV = np.geomspace(1e-8, 1e12, 201)
jet = BHJet(verbosity_level=0)
jet.init_jet_dynamics(BLJet(verbosity_level=0))

start = time.perf_counter()
jet.compute_full_jet((energy_eV / 6.241509074e11).tolist())
elapsed = time.perf_counter() - start

print(json.dumps({
    "time": elapsed,
    "energy": energy_eV.tolist(),
    "total": jet.get_observed_photon_flux_total(),
    "synchrotron": jet.get_observed_photon_integrated_flux_electron_cyclosyn(0, float("inf")),
    "compton": jet.get_observed_photon_integrated_flux_electron_compton(0, float("inf")),
}))
"""

def run_model(root, threads):
    environment = dict(
        os.environ,
        PYTHONPATH=str(root / "python"),
        OMP_NUM_THREADS=str(threads),
    )
    output = subprocess.check_output(
        [sys.executable, "-c", calculation], env=environment, text=True
    )
    return json.loads(output.splitlines()[-1])


In [ ]:
serial = run_model(serial_root, 1)
one_thread = run_model(openmp_root, 1)
four_threads = run_model(openmp_root, 4)

for name, result in [("Serial", serial), ("OpenMP: 1 thread", one_thread),
                     ("OpenMP: 4 threads", four_threads)]:
    print(f"{name}: {result['time']:.3f} s")

print(f"OpenMP speed-up: {one_thread['time'] / four_threads['time']:.2f}x")

for component in ["total", "synchrotron", "compton"]:
    serial_matches = np.allclose(serial[component], one_thread[component], rtol=1e-12, atol=0)
    threads_match = np.allclose(one_thread[component], four_threads[component], rtol=1e-12, atol=0)
    print(f"{component}: serial vs OpenMP = {serial_matches}, 1 vs 4 threads = {threads_match}")


In [ ]:
fig, (ax_spectrum, ax_time) = plt.subplots(1, 2, figsize=(11, 4))

for label, result in [("Serial", serial), ("OpenMP: 1 thread", one_thread),
                      ("OpenMP: 4 threads", four_threads)]:
    ax_spectrum.loglog(result["energy"], result["total"], label=label)

ax_spectrum.set_xlabel("Energy [eV]")
ax_spectrum.set_ylabel(r"Photon flux [cm$^{-2}$ s$^{-1}$]")
ax_spectrum.legend()

ax_time.bar(
    ["Serial", "1 thread", "4 threads"],
    [serial["time"], one_thread["time"], four_threads["time"]],
)
ax_time.set_ylabel("Calculation time [s]")

plt.tight_layout()
plt.show()
